<a href="https://colab.research.google.com/github/csu-techhub/Quantum-Optimization-Simulation/blob/main/module12/Lab5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module12/Lab5.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 5 — Complete 5-Node Max-Cut QAOA (Instructor)
**Quantum Optimization and Simulation — QAOA Laboratory Series**

Instructor version with completed exercises and answer key.

**Format:** 10–15 minute instructor walkthrough + about 45–60 minutes of independent work.

**Notebook style:** Most code is supplied. Cells marked **YOUR TURN** contain a small value, line, or function for you to complete.

> Qiskit displays measured bitstrings in the order `q_(n-1)...q_0`. When we discuss graph nodes, this notebook often converts them to `q_0...q_(n-1)` using `q0_first(...)`.

## Learning goals
- Build the complete 5-node Max-Cut QAOA circuit used in the lecture.
- Decode measured bitstrings and calculate their cut values.
- Compare your gate-level construction with Qiskit's built-in QAOA workflow.

The graph is

\[
E=\{(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)\}.
\]

The maximum cut value is 6; complementary bitstrings represent the same partition.

In [ ]:
# Run this once at the beginning of a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-optimization~=0.7" "qiskit-ibm-runtime~=0.46"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2

SEED = 123
SHOTS = 2048
sampler = SamplerV2(default_shots=SHOTS, seed=SEED)

def run_counts(qc, shots=SHOTS):
    "Run a measured circuit with Aer SamplerV2 and return counts."
    result = sampler.run([qc], shots=shots).result()
    return result[0].data.meas.get_counts()

def q0_first(qiskit_bitstring):
    "Convert Qiskit's displayed q_(n-1)...q_0 bitstring to q_0...q_(n-1)."
    return qiskit_bitstring.replace(" ", "")[::-1]

In [ ]:
import networkx as nx

edges5 = [(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)]
G = nx.Graph()
G.add_nodes_from(range(5))
G.add_edges_from(edges5)

pos = {0:(0,2), 1:(0,1), 2:(0,0), 3:(1,1.5), 4:(1,0.5)}
nx.draw(G, pos=pos, with_labels=True, node_size=900)
plt.show()

In [ ]:
def cut_value_q0first(bitstring, edges):
    return sum(bitstring[i] != bitstring[j] for i, j in edges)

def expected_cut_from_counts(counts, edges):
    total = sum(counts.values())
    return sum(
        cut_value_q0first(q0_first(k), edges) * v / total
        for k, v in counts.items()
    )

def build_qaoa_5(gamma, beta, measure=True):
    qc = QuantumCircuit(5)
    qc.h(range(5))

    for i, j in edges5:
        qc.cx(i, j)
        qc.rz(-gamma, j)
        qc.cx(i, j)

    for q in range(5):
        qc.rx(2 * beta, q)

    if measure:
        qc.measure_all()
    return qc

## Part A — Run one p=1 circuit

In [ ]:
gamma = 0.68
beta = 0.39

qc = build_qaoa_5(gamma, beta)
display(qc.draw("mpl"))

counts = run_counts(qc, shots=4096)
print("Expected cut =", expected_cut_from_counts(counts, edges5))
plot_histogram(counts)

**Expected qualitative result:** the average cut should be noticeably above the uniform-random value of 3, although a single \(p=1\) layer is not expected to reproduce the deeper optimized lecture result.

## Part B — Rank the measured candidates

In [ ]:
ranked = []
for qiskit_bits, count in counts.items():
    bits = q0_first(qiskit_bits)
    ranked.append((bits, count, cut_value_q0first(bits, edges5)))

ranked = sorted(ranked, key=lambda x: (x[2], x[1]), reverse=True)
ranked[:10]

### YOUR TURN
Find all bitstrings with cut value 6.

In [ ]:
best_strings = [
    bits for bits, count, cut in ranked
    if cut == 0   # TODO: replace 0 with the maximum cut value
]
best_strings

**Expected:** the optimal partition appears as the complementary pair `00011` and `11100` when bitstrings are written q0→q4.

## Part C — Compare with Qiskit's built-in QAOA

In [ ]:
from qiskit_optimization.applications import Maxcut
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.minimum_eigensolvers import QAOA
from qiskit_optimization.optimizers import COBYLA
from qiskit_aer.primitives import SamplerV2

maxcut = Maxcut(G)
qp = maxcut.to_quadratic_program()
print(qp.prettyprint())

builtin_sampler = SamplerV2(default_shots=2048, seed=SEED)
builtin_qaoa = QAOA(
    sampler=builtin_sampler,
    optimizer=COBYLA(maxiter=60),
    reps=1
)

optimizer = MinimumEigenOptimizer(builtin_qaoa)
result = optimizer.solve(qp)
print(result.prettyprint())
print("Selected partition vector:", result.x)

**Expected:** Qiskit's high-level QAOA should return a feasible high-quality Max-Cut solution. Because QAOA is stochastic and the optimizer is approximate, the exact bitstring and objective value can vary.

## Reflection
1. Which gates in your manual circuit correspond to \(U_C\)?
2. Which gates correspond to \(U_B\)?
3. What does the high-level `QAOA(...)` call hide from the learner?

## Instructor solutions

In [ ]:
best_strings = [
    bits for bits, count, cut in ranked
    if cut == 6
]
best_strings

1. \(U_C\) is the six edge blocks, each implemented as `CX–RZ(-gamma)–CX`.
2. \(U_B\) is the set of five `RX(2*beta)` gates.
3. The high-level call hides construction of the QAOA ansatz, parameter binding, repeated sampler calls, and the classical optimization loop.